# 🚀 Optimized RAG System with Ollama
This notebook implements a production-ready RAG (Retrieval-Augmented Generation) system with improved performance and output quality.

In [ ]:
# Install required packages
!pip install -q langchain langchain-community langchain-text-splitters pypdf faiss-cpu sentence-transformers

In [ ]:
# Import all required libraries
import os
import sys
from typing import List, Dict
import time
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import Ollama
from langchain.schema import Document

print("✅ All libraries imported successfully!")

In [ ]:
# Configuration and Global Variables
CONFIG = {
    'EMBEDDING_MODEL': 'sentence-transformers/all-MiniLM-L6-v2',  # Fastest model
    'CHUNK_SIZE': 800,          # Reduced for better relevance
    'CHUNK_OVERLAP': 150,       # Better context preservation
    'RETRIEVAL_K': 4,           # Retrieve top 4 chunks
    'OLLAMA_MODEL': 'mistral',  # Use Mistral for better quality
    'TEMPERATURE': 0.3,         # Lower for more focused answers
    'MAX_TOKENS': 512,          # Reasonable response length
}

# Global state management
class RAGSystem:
    def __init__(self):
        self.embedding_model = None
        self.vector_db = None
        self.ollama_model = None
        self.documents = []
        self.chunks = []
        self.is_initialized = False
        
    def check_ollama_connection(self) -> bool:
        """Check if Ollama is running"""
        try:
            import subprocess
            result = subprocess.run(['curl', '-s', 'http://localhost:11434/api/tags'], 
                                  capture_output=True, timeout=5)
            return result.returncode == 0
        except:
            return False
    
    def initialize_embeddings(self):
        """Initialize embedding model once"""
        if self.embedding_model is None:
            print("📥 Loading embedding model...")
            self.embedding_model = HuggingFaceEmbeddings(
                model_name=CONFIG['EMBEDDING_MODEL'],
                model_kwargs={'trust_remote_code': True},
                encode_kwargs={'normalize_embeddings': True}
            )
            print("✅ Embedding model loaded!")
    
    def initialize_ollama(self):
        """Initialize Ollama model"""
        if not self.check_ollama_connection():
            print("⚠️  Ollama is not running! Please start Ollama first.")
            print("On Colab: Run this command in a terminal:")
            print("  curl https://ollama.ai/install.sh | sh && ollama serve")
            return False
            
        if self.ollama_model is None:
            print(f"🤖 Initializing {CONFIG['OLLAMA_MODEL']} model...")
            self.ollama_model = Ollama(
                model=CONFIG['OLLAMA_MODEL'],
                temperature=CONFIG['TEMPERATURE']
            )
            print("✅ Ollama model ready!")
        return True

# Create global RAG system instance
rag_system = RAGSystem()
print("✅ RAG System initialized!")

In [ ]:
def load_documents(pdf_paths: List[str]) -> List[Document]:
    """Load PDF documents efficiently"""
    documents = []
    
    print(f"📂 Loading {len(pdf_paths)} PDF documents...")
    
    for idx, path in enumerate(pdf_paths, 1):
        if not os.path.exists(path):
            print(f"⚠️  File not found: {path}")
            continue
            
        try:
            loader = PyPDFLoader(path)
            docs = loader.load()
            documents.extend(docs)
            print(f"  ✅ [{idx}] {path} → {len(docs)} pages")
        except Exception as e:
            print(f"  ❌ [{idx}] {path} → Error: {e}")
    
    print(f"\n📊 Total pages loaded: {len(documents)}")
    return documents

def split_documents(documents: List[Document]) -> List[Document]:
    """Split documents into optimized chunks"""
    print(f"\n✂️  Splitting documents into chunks...")
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CONFIG['CHUNK_SIZE'],
        chunk_overlap=CONFIG['CHUNK_OVERLAP'],
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    chunks = splitter.split_documents(documents)
    print(f"📦 Total chunks created: {len(chunks)}")
    
    # Show sample chunks
    print(f"\n🔍 Sample chunks (first 3):")
    for i in range(min(3, len(chunks))):
        content = chunks[i].page_content[:300].replace('\n', ' ')
        print(f"\n  [{i+1}] {content}...")
    
    return chunks

def create_vector_store(chunks: List[Document]):
    """Create and store vector embeddings"""
    print(f"\n🧠 Creating vector database...")
    
    # Initialize embeddings
    rag_system.initialize_embeddings()
    
    # Create FAISS vector store with batch processing
    start_time = time.time()
    rag_system.vector_db = FAISS.from_documents(
        chunks, 
        rag_system.embedding_model,
        distance_strategy="COSINE"
    )
    elapsed = time.time() - start_time
    
    print(f"✅ Vector database created!")
    print(f"  📈 Total vectors: {rag_system.vector_db.index.ntotal}")
    print(f"  ⏱️  Time taken: {elapsed:.2f}s")
    print(f"  💾 Vector dimension: 384")

# Example usage
pdf_paths = [
    "/content/pdf1.pdf",
    "/content/pdf2.pdf",
    "/content/pdf3.pdf",
    "/content/pdf4.pdf",
    "/content/pdf5.pdf",
]

# Load and process documents
documents = load_documents(pdf_paths)

if documents:
    chunks = split_documents(documents)
    create_vector_store(chunks)
    rag_system.chunks = chunks
    print("\n✅ Document processing complete!")
else:
    print("\n❌ No documents loaded. Please check PDF paths.")

In [ ]:
def retrieve_context(query: str, k: int = None) -> tuple:
    """Retrieve relevant documents for a query"""
    if k is None:
        k = CONFIG['RETRIEVAL_K']
    
    if rag_system.vector_db is None:
        print("❌ Vector database not initialized. Please run document loading first.")
        return None, None
    
    print(f"\n🔎 Searching for: '{query}'")
    results = rag_system.vector_db.similarity_search_with_scores(query, k=k)
    
    # Prepare context
    context_parts = []
    documents = []
    
    print(f"\n📄 Top {len(results)} retrieved documents:")
    for i, (doc, score) in enumerate(results, 1):
        documents.append(doc)
        context_parts.append(f"[Document {i}]\n{doc.page_content}")
        
        content_preview = doc.page_content[:200].replace('\n', ' ')
        print(f"  [{i}] Similarity: {score:.3f} → {content_preview}...")
    
    context = "\n\n---\n\n".join(context_parts)
    return context, documents

# Test retrieval with a sample query
if rag_system.vector_db is not None:
    test_query = "What is the main topic of these documents?"
    context, docs = retrieve_context(test_query)
    
    if context:
        print(f"\n✅ Context retrieved successfully!")
        print(f"   Total context length: {len(context)} characters")

In [ ]:
def generate_rag_answer(query: str, context: str = None) -> str:
    """Generate answer using RAG (Retrieval-Augmented Generation)"""
    
    # Initialize Ollama if needed
    if not rag_system.initialize_ollama():
        return "❌ Ollama not available"
    
    # Retrieve context if not provided
    if context is None:
        context, _ = retrieve_context(query)
        if context is None:
            return "❌ Failed to retrieve context"
    
    # Craft optimized prompt
    prompt = f"""You are a knowledgeable assistant helping users understand documents.

Instructions:
1. Answer ONLY based on the provided context
2. Be concise and direct
3. If the answer is not in the context, say: "I cannot find this information in the documents."
4. Cite which document provided the answer if possible
5. Do not make up information

Context from documents:
{context}

User Question: {query}

Answer:"""

    try:
        print(f"\n🤖 Generating answer...")
        response = rag_system.ollama_model.invoke(prompt)
        return response.strip()
    except Exception as e:
        return f"❌ Error: {str(e)}"

def generate_standard_answer(query: str) -> str:
    """Generate answer WITHOUT RAG for comparison"""
    
    if not rag_system.initialize_ollama():
        return "❌ Ollama not available"
    
    prompt = f"""Answer the following question concisely:

Question: {query}

Answer:"""
    
    try:
        print(f"\n🧠 Generating standard answer...")
        response = rag_system.ollama_model.invoke(prompt)
        return response.strip()
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Test with a sample query
if rag_system.vector_db is not None:
    test_query = "Summarize the key points mentioned in the documents."
    
    print("\n" + "="*70)
    print("TESTING RAG SYSTEM")
    print("="*70)
    
    # RAG Answer
    print(f"\n📝 Query: {test_query}")
    print("\n" + "-"*70)
    print("WITH CONTEXT (RAG):")
    print("-"*70)
    context, _ = retrieve_context(test_query, k=3)
    rag_answer = generate_rag_answer(test_query, context)
    print(f"\n{rag_answer}")
    
    # Standard Answer for comparison
    print("\n" + "-"*70)
    print("WITHOUT CONTEXT (Standard LLM):")
    print("-"*70)
    standard_answer = generate_standard_answer(test_query)
    print(f"\n{standard_answer}")

In [ ]:
def interactive_rag_chat():
    """Interactive multi-turn RAG conversation"""
    
    # Check initialization
    if rag_system.vector_db is None:
        print("❌ Vector database not initialized.")
        print("   Please run the document loading and processing cells first.")
        return
    
    if not rag_system.initialize_ollama():
        print("❌ Ollama not available. Cannot start chat.")
        return
    
    print("\n" + "="*70)
    print("🤖 INTERACTIVE RAG CHAT SYSTEM")
    print("="*70)
    print("\n💡 Instructions:")
    print("   • Ask questions about the documents")
    print("   • Type 'exit', 'quit', or 'q' to end the conversation")
    print("   • Type 'stats' to see system statistics")
    print("   • Type 'clear' to reset conversation history")
    print("\n" + "="*70 + "\n")
    
    conversation_history = []
    
    while True:
        try:
            # Get user input
            user_input = input("\n💬 You: ").strip()
            
            # Handle empty input
            if not user_input:
                print("   ⓘ Please enter a question.")
                continue
            
            # Handle special commands
            if user_input.lower() in ['exit', 'quit', 'q']:
                print("\n👋 Thank you for using RAG Chat. Goodbye!")
                break
            
            if user_input.lower() == 'stats':
                print(f"\n📊 System Statistics:")
                print(f"   • Vector Database Size: {rag_system.vector_db.index.ntotal}")
                print(f"   • Embedding Model: {CONFIG['EMBEDDING_MODEL']}")
                print(f"   • LLM Model: {CONFIG['OLLAMA_MODEL']}")
                print(f"   • Conversation Turns: {len(conversation_history)}")
                continue
            
            if user_input.lower() == 'clear':
                conversation_history = []
                print("\n🧹 Conversation history cleared.")
                continue
            
            # Process query with RAG
            print(f"\n🔍 Searching and thinking...")
            
            context, retrieved_docs = retrieve_context(user_input, k=CONFIG['RETRIEVAL_K'])
            
            if context is None:
                print("\n❌ Failed to retrieve context.")
                continue
            
            # Generate response
            response = generate_rag_answer(user_input, context)
            
            # Display response
            print("\n" + "="*70)
            print(f"🤖 Bot: {response}")
            print("="*70)
            
            # Store in history
            conversation_history.append({
                "question": user_input,
                "answer": response,
                "context_used": len(retrieved_docs)
            })
            
        except KeyboardInterrupt:
            print("\n\n👋 Chat interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            print("   Please try again.")

# Run interactive chat
if __name__ == "__main__":
    interactive_rag_chat()

In [ ]:
def batch_query_documents(queries: List[str]) -> Dict:
    """Process multiple queries at once"""
    
    if rag_system.vector_db is None:
        print("❌ Vector database not initialized.")
        return {}
    
    if not rag_system.initialize_ollama():
        print("❌ Ollama not available.")
        return {}
    
    results = {}
    
    print("\n" + "="*70)
    print(f"📋 BATCH QUERY PROCESSING ({len(queries)} queries)")
    print("="*70)
    
    for idx, query in enumerate(queries, 1):
        print(f"\n[{idx}/{len(queries)}] Processing: {query[:50]}...")
        
        context, docs = retrieve_context(query, k=3)
        if context:
            answer = generate_rag_answer(query, context)
            results[query] = {
                "answer": answer,
                "documents_used": len(docs)
            }
            print(f"   ✅ Completed")
        else:
            results[query] = {
                "answer": "Failed to retrieve context",
                "documents_used": 0
            }
            print(f"   ❌ Failed")
    
    return results

# Example batch query
if rag_system.vector_db is not None:
    sample_queries = [
        "What is the main topic?",
        "Who are the key entities mentioned?",
        "What are the important dates mentioned?",
    ]
    
    batch_results = batch_query_documents(sample_queries)
    
    # Display results
    if batch_results:
        print("\n" + "="*70)
        print("BATCH RESULTS SUMMARY")
        print("="*70)
        for query, result in batch_results.items():
            print(f"\nQ: {query}")
            print(f"A: {result['answer'][:200]}...")
            print(f"   📄 Documents used: {result['documents_used']}")

In [ ]:
def export_results(queries: List[str], filename: str = "rag_results.txt"):
    """Export RAG results to file"""
    
    if rag_system.vector_db is None:
        print("❌ Vector database not initialized.")
        return
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write("RAG SYSTEM RESULTS REPORT\n")
        f.write("="*70 + "\n\n")
        
        f.write(f"Configuration:\n")
        for key, value in CONFIG.items():
            f.write(f"  • {key}: {value}\n")
        f.write("\n")
        
        results = batch_query_documents(queries)
        
        for idx, (query, result) in enumerate(results.items(), 1):
            f.write(f"\nQuery {idx}: {query}\n")
            f.write("-"*70 + "\n")
            f.write(f"Answer:\n{result['answer']}\n")
            f.write(f"\nDocuments Used: {result['documents_used']}\n")
            f.write("\n")
    
    print(f"✅ Results exported to {filename}")

# Example export
if rag_system.vector_db is not None:
    export_queries = [
        "What is the primary subject of these documents?",
        "List the main concepts discussed.",
        "What are the conclusions or recommendations?",
    ]
    
    # Uncomment to export:
    # export_results(export_queries)